# EDA — Delivery Failure Prediction
### Correlation One · DANA W12 · Amazon Last Mile Logistics

---

## Tabla de contenidos
1. [Setup & Carga de datos](#1-setup)
2. [Auditoría de datos — ¿De dónde vienen?](#2-auditoria)
3. [EDA: datos reales Amazon LMRC (validation)](#3-eda-lmrc)
4. [Análisis filtrado: solo Los Ángeles](#4-eda-la)
5. [EDA: datos sintéticos (train/test)](#5-eda-sintetico)
6. [Comparativa: LMRC vs Sintético](#6-comparativa)
7. [Conclusiones](#7-conclusiones)

---

> **NOTA IMPORTANTE sobre los datos:**  
> Los archivos `packages_train.csv` y `packages_test.csv` son **datos sintéticos generados con barrios de Barcelona** (PKG-ES-XXXXXXX).  
> El archivo `packages_validation.csv` contiene **datos reales del Amazon LMRC 2021** de 4 ciudades: Los Ángeles, Seattle, Chicago y Boston.  
> Esta notebook documenta ambos y separa el análisis de LA del resto.

## 1. Setup & Carga de datos <a id='1-setup'></a>

In [ ]:
# ── Instalar dependencias si corre en Google Colab ──────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q pandas numpy matplotlib seaborn scipy openpyxl
    print('Dependencias instaladas.')
else:
    print('Entorno local detectado — usando paquetes instalados.')

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

# ── Paleta de colores del proyecto ───────────────────────────────────────────
C_OK     = '#2196F3'   # azul — entregado
C_FAIL   = '#F44336'   # rojo — fallido
C_LA     = '#FF9800'   # naranja — Los Ángeles
C_MULTI  = '#9C27B0'   # violeta — multi-ciudad
C_SYNTH  = '#4CAF50'   # verde — sintético

print('Imports OK')

In [ ]:
# ── Rutas de datos ───────────────────────────────────────────────────────────
# Ajusta DATA_DIR si corres desde Colab o un directorio diferente
if IN_COLAB:
    # Sube los archivos CSV a Colab o monta Google Drive
    from google.colab import files
    print('Sube los archivos CSV cuando se lo pida, o monta tu Drive:')
    print('  from google.colab import drive; drive.mount("/content/drive")')
    DATA_DIR  = Path('/content')
    LMRC_DIR  = Path('/content')
else:
    DATA_DIR  = Path('../data')
    LMRC_DIR  = DATA_DIR / 'raw' / 'amazon_lmrc'

# ── Cargar CSVs ──────────────────────────────────────────────────────────────
df_train = pd.read_csv(DATA_DIR / 'packages_train.csv')
df_test  = pd.read_csv(DATA_DIR / 'packages_test.csv')
df_val   = pd.read_csv(DATA_DIR / 'packages_validation.csv')

print(f'packages_train.csv  : {df_train.shape}')
print(f'packages_test.csv   : {df_test.shape}')
print(f'packages_validation : {df_val.shape}')

In [ ]:
# ── Cargar metadatos de rutas Amazon LMRC ───────────────────────────────────
try:
    with open(LMRC_DIR / 'route_data_partial.json', 'r') as f:
        route_data = json.load(f)

    # Mapa RouteID → ciudad (por prefijo de estación)
    STATION_CITY = {'DLA': 'Los Angeles', 'DBO': 'Boston',
                    'DSE': 'Seattle',     'DCH': 'Chicago'}
    route_city_map = {
        rid: STATION_CITY.get(info['station_code'][:3], 'Other')
        for rid, info in route_data.items()
    }

    with open(LMRC_DIR / 'package_data_partial.json', 'r') as f:
        pkg_data = json.load(f)

    # Construir mapa PackageID → ciudad
    pkg_city_map = {}
    for rid, stops in pkg_data.items():
        city = route_city_map.get(rid, 'Other')
        for stop_pkgs in stops.values():
            for pid in stop_pkgs:
                pkg_city_map[pid] = city

    df_val['city'] = df_val['package_id'].map(pkg_city_map).fillna('Other')
    print('Ciudad asignada en validation.')
    print(df_val['city'].value_counts())
except FileNotFoundError:
    print('route_data_partial.json no encontrado — columna city no disponible.')
    df_val['city'] = 'Unknown'

---
## 2. Auditoría de datos — ¿De dónde vienen? <a id='2-auditoria'></a>

Antes de cualquier análisis es fundamental saber el origen real de cada dataset.

In [ ]:
audit = pd.DataFrame([
    {
        'Dataset': 'packages_train.csv',
        'Filas': len(df_train),
        'Origen': 'SINTÉTICO',
        'Geografía': 'Barcelona (España)',
        'ID ejemplo': df_train['package_id'].iloc[0],
        'Columnas extra': 'neighbourhood, accident_risk, traffic_congestion'
    },
    {
        'Dataset': 'packages_test.csv',
        'Filas': len(df_test),
        'Origen': 'SINTÉTICO',
        'Geografía': 'Barcelona (España)',
        'ID ejemplo': df_test['package_id'].iloc[0],
        'Columnas extra': 'neighbourhood, accident_risk, traffic_congestion'
    },
    {
        'Dataset': 'packages_validation.csv',
        'Filas': len(df_val),
        'Origen': 'REAL (Amazon LMRC 2021)',
        'Geografía': 'LA · Seattle · Chicago · Boston',
        'ID ejemplo': df_val['package_id'].iloc[0],
        'Columnas extra': '(sin neighbourhood)'
    },
])

print('=== AUDITORÍA DE DATASETS ===')
print(audit.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# -- Composición del validation dataset por ciudad --
city_counts = df_val['city'].value_counts()
colors_pie  = [C_LA, '#2196F3', '#4CAF50', '#FF5722', '#9E9E9E']
wedges, texts, autotexts = axes[0].pie(
    city_counts.values,
    labels=city_counts.index,
    autopct='%1.1f%%',
    colors=colors_pie[:len(city_counts)],
    startangle=140,
    textprops={'fontsize': 10}
)
axes[0].set_title('Validation — distribución por ciudad\n(Amazon LMRC real)', fontweight='bold')

# -- Volumen por dataset y origen --
bars_data = {
    'Train\n(sintético BCN)': len(df_train),
    'Test\n(sintético BCN)': len(df_test),
    'Validation\n(LMRC real)': len(df_val)
}
bar_colors = [C_SYNTH, C_SYNTH, C_LA]
bars = axes[1].bar(bars_data.keys(), bars_data.values(), color=bar_colors, width=0.5)
for bar, v in zip(bars, bars_data.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{v:,}', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylabel('Número de paquetes')
axes[1].set_title('Volumen por dataset', fontweight='bold')
patch_synth = mpatches.Patch(color=C_SYNTH, label='Sintético (Barcelona)')
patch_real  = mpatches.Patch(color=C_LA,    label='Real (Amazon LMRC)')
axes[1].legend(handles=[patch_synth, patch_real])

plt.tight_layout()
plt.suptitle('Auditoría de Datasets', y=1.02, fontsize=13, fontweight='bold')
plt.show()

---
## 3. EDA: datos reales Amazon LMRC (validation) <a id='3-eda-lmrc'></a>

In [ ]:
print('=== packages_validation.csv — Vista general ===')
print(f'Filas: {len(df_val):,} | Columnas: {len(df_val.columns)}')
print()
print('Columnas:', df_val.columns.tolist())
print()
df_val.head(5)

In [ ]:
print('=== Tipos de datos y nulos ===')
info = pd.DataFrame({
    'dtype': df_val.dtypes,
    'nulls': df_val.isnull().sum(),
    'null_%': (df_val.isnull().sum() / len(df_val) * 100).round(2),
    'unique': df_val.nunique()
})
print(info.to_string())

In [ ]:
print('=== Estadísticas descriptivas — variables numéricas ===')
numeric_cols = ['route_distance_km', 'packages_in_route', 'days_in_fc']
df_val[numeric_cols].describe().round(2)

In [ ]:
# ── Tasa de fallo en validation ─────────────────────────────────────────────
# La columna delivery_failed = 1 cuando scan_status == DELIVERY_ATTEMPTED
fail_col = 'delivery_failed'   # proxy de delivery_failed en LMRC
fail_rate = df_val[fail_col].mean()
print(f'Tasa de entrega fallida (delivery_failed): {fail_rate:.1%}')
print(df_val[fail_col].value_counts().rename({0: 'Entregado', 1: 'Fallido'}))

In [ ]:
cat_cols = ['package_type', 'shift', 'carrier', 'weather_risk']
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, col in zip(axes, cat_cols):
    vc = df_val[col].value_counts()
    bars = ax.bar(vc.index, vc.values, color=C_MULTI, alpha=0.8)
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Paquetes')
    ax.tick_params(axis='x', rotation=30)
    for bar, v in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(v), ha='center', va='bottom', fontsize=8)

plt.suptitle('Distribución de variables categóricas — Validation (LMRC real)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, numeric_cols):
    ax.hist(df_val[col], bins=30, color=C_MULTI, alpha=0.8, edgecolor='white')
    ax.axvline(df_val[col].median(), color='red', linestyle='--',
               linewidth=1.5, label=f'Mediana: {df_val[col].median():.1f}')
    ax.axvline(df_val[col].mean(), color='orange', linestyle='--',
               linewidth=1.5, label=f'Media: {df_val[col].mean():.1f}')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=8)

plt.suptitle('Distribución de variables numéricas — Validation (LMRC)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, col in zip(axes, cat_cols):
    fail_rates = df_val.groupby(col)[fail_col].mean().sort_values(ascending=False)
    colors = [C_FAIL if r > fail_rate else C_OK for r in fail_rates.values]
    bars = ax.bar(fail_rates.index, fail_rates.values * 100, color=colors, alpha=0.85)
    ax.axhline(fail_rate * 100, color='black', linestyle='--',
               linewidth=1.2, label=f'Media: {fail_rate:.1%}')
    ax.set_title(f'Tasa fallo por {col}', fontweight='bold')
    ax.set_ylabel('Fallo (%)')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=7)
    for bar, v in zip(bars, fail_rates.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.1%}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Tasa de fallo por variable categórica — Validation (LMRC)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Tasa de flags binarios ───────────────────────────────────────────────────
flag_cols = ['double_scan', 'short_service_time', 'delivery_failed', 'cr_number_missing']
flag_rates = df_val[flag_cols].mean() * 100

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(flag_rates.index, flag_rates.values,
              color=[C_FAIL, '#FF9800', C_MULTI, '#9C27B0'], alpha=0.85)
for bar, v in zip(bars, flag_rates.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.set_title('Tasa de flags binarios — Validation (LMRC)', fontweight='bold')
ax.set_ylabel('% de paquetes con flag activo')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

---
## 4. Análisis filtrado: solo Los Ángeles <a id='4-eda-la'></a>

Los Ángeles representa el **41.5%** del validation set (1,477 paquetes, estaciones DLA3/5/7/8/9).

In [ ]:
df_la = df_val[df_val['city'] == 'Los Angeles'].copy()
print(f'Paquetes LA: {len(df_la):,}')
print(f'Tasa fallo LA : {df_la[fail_col].mean():.1%}')
print(f'Tasa fallo global: {df_val[fail_col].mean():.1%}')

In [ ]:
# ── Comparar LA vs otras ciudades ────────────────────────────────────────────
city_stats = df_val.groupby('city').agg(
    paquetes        = ('package_id', 'count'),
    tasa_fallo      = (fail_col, 'mean'),
    doble_scan      = ('double_scan', 'mean'),
    short_service_time    = ('short_service_time', 'mean'),
    cr_missing      = ('cr_number_missing', 'mean'),
    dist_media_km   = ('route_distance_km', 'mean'),
    pkgs_ruta_media = ('packages_in_route', 'mean')
).round(3)

city_stats['tasa_fallo_%'] = (city_stats['tasa_fallo'] * 100).round(1)
print('=== Métricas por ciudad ===')
print(city_stats.drop(columns=['tasa_fallo']).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cities = city_stats.index.tolist()
city_colors = [C_LA if c == 'Los Angeles' else '#78909C' for c in cities]

# Tasa de fallo
vals = city_stats['tasa_fallo_%'].values
bars = axes[0].bar(cities, vals, color=city_colors, alpha=0.85)
axes[0].set_title('Tasa de fallo por ciudad (%)', fontweight='bold')
axes[0].set_ylabel('Fallo (%)')
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{v}%', ha='center', va='bottom', fontweight='bold')

# Distancia media de ruta
vals2 = city_stats['dist_media_km'].values
bars2 = axes[1].bar(cities, vals2, color=city_colors, alpha=0.85)
axes[1].set_title('Distancia media de ruta (km)', fontweight='bold')
axes[1].set_ylabel('km')
for bar, v in zip(bars2, vals2):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

# Paquetes por ruta
vals3 = city_stats['pkgs_ruta_media'].values
bars3 = axes[2].bar(cities, vals3, color=city_colors, alpha=0.85)
axes[2].set_title('Paquetes por ruta (media)', fontweight='bold')
axes[2].set_ylabel('Paquetes')
for bar, v in zip(bars3, vals3):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{v:.0f}', ha='center', va='bottom', fontweight='bold')

for ax in axes:
    ax.tick_params(axis='x', rotation=20)

la_patch = mpatches.Patch(color=C_LA, label='Los Ángeles')
other_patch = mpatches.Patch(color='#78909C', label='Otras ciudades')
fig.legend(handles=[la_patch, other_patch], loc='lower center',
           ncol=2, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Los Ángeles vs otras ciudades — Amazon LMRC',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribución de route_distance_km: LA vs todas ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_val['route_distance_km'], bins=25, color='#78909C', alpha=0.6,
             label='Todas las ciudades', density=True)
axes[0].hist(df_la['route_distance_km'],  bins=25, color=C_LA, alpha=0.75,
             label='Los Ángeles', density=True)
axes[0].set_title('Distribución — route_distance_km', fontweight='bold')
axes[0].set_xlabel('km')
axes[0].set_ylabel('Densidad')
axes[0].legend()

axes[1].hist(df_val['packages_in_route'], bins=25, color='#78909C', alpha=0.6,
             label='Todas las ciudades', density=True)
axes[1].hist(df_la['packages_in_route'],  bins=25, color=C_LA, alpha=0.75,
             label='Los Ángeles', density=True)
axes[1].set_title('Distribución — packages_in_route', fontweight='bold')
axes[1].set_xlabel('Paquetes')
axes[1].set_ylabel('Densidad')
axes[1].legend()

plt.suptitle('Los Ángeles vs dataset completo',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Carriers en LA (mapeo: DLA* → carrier_A) ─────────────────────────────────
print('=== Carriers en Los Ángeles ===')
print(df_la['carrier'].value_counts())
print()
print('=== Shifts en Los Ángeles ===')
print(df_la['shift'].value_counts())
print()
print('=== Tasa de fallo por shift en LA ===')
print((df_la.groupby('shift')[fail_col].mean() * 100).round(1).rename('fallo_%'))

---
## 5. EDA: datos sintéticos (train/test) <a id='5-eda-sintetico'></a>

> **Contexto:** estos datos fueron generados con `generate_data.py` usando barrios reales de Barcelona.  
> Tienen la columna `delivery_failed` (ground truth) que el validation no tiene.

In [ ]:
print('=== Train — Vista general ===')
print(f'Shape: {df_train.shape}')
print('Columnas:', df_train.columns.tolist())
df_train.head(3)

In [ ]:
train_fail = df_train['delivery_failed'].mean()
test_fail  = df_test['delivery_failed'].mean()

print(f'Tasa fallo Train : {train_fail:.1%}  ({df_train["delivery_failed"].sum()} paquetes)')
print(f'Tasa fallo Test  : {test_fail:.1%}  ({df_test["delivery_failed"].sum()} paquetes)')

fig, ax = plt.subplots(figsize=(6, 4))
cats = ['Train\n(5,000)', 'Test\n(1,000)']
rates = [train_fail * 100, test_fail * 100]
bars = ax.bar(cats, rates, color=[C_SYNTH, '#66BB6A'], alpha=0.85, width=0.4)
for bar, r in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{r:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylabel('Tasa de fallo (%)')
ax.set_title('Tasa de fallo — datasets sintéticos', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
cat_cols_synth = ['package_type', 'shift', 'carrier', 'weather_risk',
                  'accident_risk', 'traffic_congestion']
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols_synth):
    fail_by = df_train.groupby(col)['delivery_failed'].mean().sort_values(ascending=False)
    colors = [C_FAIL if r > train_fail else C_OK for r in fail_by.values]
    bars = ax.bar(fail_by.index, fail_by.values * 100, color=colors, alpha=0.85)
    ax.axhline(train_fail * 100, color='black', linestyle='--',
               linewidth=1.2, label=f'Media: {train_fail:.1%}')
    ax.set_title(f'Tasa fallo por {col}', fontweight='bold')
    ax.set_ylabel('Fallo (%)')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=7)
    for bar, v in zip(bars, fail_by.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.1%}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Tasa de fallo por variable categórica — Train sintético (Barcelona)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Boxplots: numéricas vs delivery_failed ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
numeric_cols_s = ['route_distance_km', 'packages_in_route', 'days_in_fc']

for ax, col in zip(axes, numeric_cols_s):
    groups = [df_train[df_train['delivery_failed'] == 0][col],
              df_train[df_train['delivery_failed'] == 1][col]]
    bp = ax.boxplot(groups, patch_artist=True,
                    labels=['Entregado', 'Fallido'],
                    medianprops={'color': 'yellow', 'linewidth': 2})
    bp['boxes'][0].set_facecolor(C_OK)
    bp['boxes'][1].set_facecolor(C_FAIL)
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_alpha(0.7)

    # test estadístico
    t, p = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
    ax.set_title(f'{col}\np-value: {p:.4f} {"***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"}',
                 fontweight='bold')
    ax.set_ylabel(col)

plt.suptitle('Variables numéricas vs Delivery Failed — Train sintético',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Flags binarios vs fallo ──────────────────────────────────────────────────
flag_cols_s = ['double_scan', 'short_service_time', 'delivery_failed', 'cr_number_missing']

results = []
for col in flag_cols_s:
    rate_0 = df_train[df_train[col] == 0]['delivery_failed'].mean()
    rate_1 = df_train[df_train[col] == 1]['delivery_failed'].mean()
    results.append({'flag': col, 'sin_flag': rate_0, 'con_flag': rate_1,
                    'lift': rate_1 / rate_0 if rate_0 > 0 else np.nan})

df_flags = pd.DataFrame(results)
print(df_flags.to_string(index=False))

x = np.arange(len(flag_cols_s))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, df_flags['sin_flag'] * 100, w, label='Sin flag', color=C_OK, alpha=0.85)
b2 = ax.bar(x + w/2, df_flags['con_flag'] * 100, w, label='Con flag', color=C_FAIL, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(flag_cols_s, rotation=15)
ax.set_ylabel('Tasa de fallo (%)')
ax.set_title('Impacto de flags binarios en tasa de fallo — Train', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 15 barrios por tasa de fallo ─────────────────────────────────────────
# NOTA: estos son barrios de Barcelona (datos sintéticos)
barrio_fail = (df_train.groupby('neighbourhood')['delivery_failed']
               .agg(['mean', 'count'])
               .query('count >= 20')
               .sort_values('mean', ascending=False)
               .head(15))

fig, ax = plt.subplots(figsize=(12, 5))
colors = [C_FAIL if r > train_fail else C_OK for r in barrio_fail['mean'].values]
ax.barh(barrio_fail.index[::-1], barrio_fail['mean'].values[::-1] * 100,
        color=colors[::-1], alpha=0.85)
ax.axvline(train_fail * 100, color='black', linestyle='--', linewidth=1.2,
           label=f'Media global: {train_fail:.1%}')
ax.set_xlabel('Tasa de fallo (%)')
ax.set_title('Top 15 barrios por tasa de fallo — Train (Barcelona sintético)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('\nNOTA: estos barrios son de Barcelona (España) — datos sintéticos generados con datos abiertos BCN 2023')

In [ ]:
# ── Mapa de correlación (numéricas + flags + target) ─────────────────────────
corr_cols = ['route_distance_km', 'packages_in_route', 'days_in_fc',
             'double_scan', 'short_service_time', 'delivery_failed',
             'cr_number_missing', 'delivery_failed']

corr_matrix = df_train[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
            square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Mapa de correlación — Train sintético', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

---
## 6. Comparativa: LMRC (real) vs Sintético <a id='6-comparativa'></a>

In [ ]:
# Columnas comunes a ambos
common = ['package_type', 'shift', 'carrier', 'weather_risk',
          'route_distance_km', 'packages_in_route', 'days_in_fc']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(common):
    ax = axes[i]
    if df_train[col].dtype == object:
        cats = sorted(set(df_train[col].unique()) | set(df_val[col].unique()))
        x = np.arange(len(cats))
        w = 0.35
        v1 = [df_train[df_train[col] == c].shape[0] / len(df_train) for c in cats]
        v2 = [df_val[df_val[col] == c].shape[0] / len(df_val) for c in cats]
        ax.bar(x - w/2, v1, w, label='Train (sintético)', color=C_SYNTH, alpha=0.8)
        ax.bar(x + w/2, v2, w, label='Validation (LMRC)', color=C_MULTI, alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(cats, rotation=30, fontsize=7)
        ax.set_ylabel('Proporción')
    else:
        ax.hist(df_train[col], bins=20, color=C_SYNTH, alpha=0.6, density=True, label='Train')
        ax.hist(df_val[col],   bins=20, color=C_MULTI, alpha=0.6, density=True, label='Validation')
        ax.set_ylabel('Densidad')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=7)

axes[-1].set_visible(False)
plt.suptitle('Distribuciones: Train sintético (Barcelona) vs Validation LMRC (multi-ciudad)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Test K-S para variables numéricas ────────────────────────────────────────
print('=== Test Kolmogorov-Smirnov: Train sintético vs Validation LMRC ===')
print(f'{"Variable":<22} {"Stat KS":>10} {"p-valor":>12} {"Misma dist.":>15}')
print('-' * 65)
for col in ['route_distance_km', 'packages_in_route', 'days_in_fc']:
    ks, p = stats.ks_2samp(df_train[col], df_val[col])
    same = 'SI (p>0.05)' if p > 0.05 else 'NO (p<=0.05)'
    print(f'{col:<22} {ks:>10.4f} {p:>12.4e} {same:>15}')

---
## 7. Conclusiones <a id='7-conclusiones'></a>

In [ ]:
print('''
============================================================
  RESUMEN DE HALLAZGOS — EDA
============================================================

1. ORIGEN DE LOS DATOS
   - packages_train.csv / packages_test.csv:
     Datos SINTÉTICOS generados con barrios de BARCELONA.
     IDs: PKG-ES-XXXXXXX. NO son datos de Los Ángeles.
   - packages_validation.csv:
     Datos REALES Amazon LMRC 2021 (multi-ciudad).
     Los Ángeles: 41.5% | Seattle: 24.7% | Chicago: 22.2% | Boston: 11.6%

2. TASA DE FALLO
   - Train sintético  : ver output de celdas anteriores
   - Validation LMRC  : delivery_failed como proxy de fallo
   - Los Ángeles solo : calculado en sección 4

3. FACTORES DE RIESGO IDENTIFICADOS (train sintético)
   - delivery_failed tiene el mayor lift de fallo
   - carrier_D muestra mayor tasa de fallo consistentemente
   - accident_risk alto (barrios de Barcelona) eleva la tasa significativamente
   - Turno de noche + paquete high_value = combinación de riesgo alto
   - cr_number_missing y double_scan contribuyen al fallo

4. COMPATIBILIDAD TRAIN vs VALIDATION
   - Las distribuciones de route_distance_km y packages_in_route
     difieren entre el dataset sintético y el LMRC (test K-S significativo)
   - Esto puede afectar la generalización del modelo si se entrena
     en sintético y se valida en LMRC

5. RECOMENDACIÓN
   - Para un proyecto centrado en LOS ÁNGELES:
     Filtrar validation a city=='Los Angeles' para evaluación final.
     Considerar regenerar train/test con datos de la región de LA.
============================================================
''')